# Fast Food Marketing Campaign A/B/C Test Analysis

This notebook compares three fast food marketing campaigns using weekly store-level sales data. It stays intentionally close to the kind of decision a business team would actually care about: which promotion should be scaled, and is the difference large enough to matter in practice rather than only on paper?

## Notebook Flow
- inspect campaign groups and baseline balance
- test mean differences with ANOVA and Tukey HSD
- estimate practical effect size and adjusted differences
- end with a rollout recommendation

## Business Problem

Which marketing campaign (Promotion 1, 2, or 3) generates the highest weekly sales?

## Hypotheses

H0: Mean weekly sales are equal across all promotions  
H1: At least one promotion differs

Significance level: α = 0.05

In [10]:
import pandas as pd
import numpy as np
from scipy import stats

### 1. EDA

In [2]:
df = pd.read_csv('WA_Marketing-Campaign.csv')
df.head()

,MarketID,MarketSize,LocationID,AgeOfStore,Promotion,week,SalesInThousands
0,1,Medium,1,4,3,1,33.73
1,1,Medium,1,4,3,2,35.67
2,1,Medium,1,4,3,3,29.03
3,1,Medium,1,4,3,4,39.25
4,1,Medium,2,5,2,1,27.81


In [3]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 548 entries, 0 to 547
Data columns (total 7 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   MarketID          548 non-null    int64  
 1   MarketSize        548 non-null    object 
 2   LocationID        548 non-null    int64  
 3   AgeOfStore        548 non-null    int64  
 4   Promotion         548 non-null    int64  
 5   week              548 non-null    int64  
 6   SalesInThousands  548 non-null    float64
dtypes: float64(1), int64(5), object(1)
memory usage: 30.1+ KB


In [ ]:
#Promotion Distribution
df.describe()

,MarketID,LocationID,AgeOfStore,Promotion,week,SalesInThousands
count,548.000000,548.000000,548.000000,548.000000,548.000000,548.000000
mean,5.715328,479.656934,8.503650,2.029197,2.500000,53.466204
std,2.877001,287.973679,6.638345,0.810729,1.119055,16.755216
min,1.000000,1.000000,1.000000,1.000000,1.000000,17.340000
25%,3.000000,216.000000,4.000000,1.000000,1.750000,42.545000
50%,6.000000,504.000000,7.000000,2.000000,2.500000,50.200000
75%,8.000000,708.000000,12.000000,3.000000,3.250000,60.477500
max,10.000000,920.000000,28.000000,3.000000,4.000000,99.650000


In [5]:
df['Promotion'].value_counts()

Promotion
3    188
2    188
1    172
Name: count, dtype: int64

In [17]:
#Mean Sales by Promotion
df.groupby('Promotion')['SalesInThousands'].agg(['mean','std','count'])

,mean,std,count
Promotion,,,
1,58.099012,16.553782,172
2,47.329415,15.108955,188
3,55.364468,16.766231,188


In [8]:
pd.crosstab(df['Promotion'], df['MarketSize'], normalize='index')

MarketSize,Large,Medium,Small
Promotion,,,
1,0.325581,0.558140,0.116279
2,0.340426,0.574468,0.085106
3,0.255319,0.617021,0.127660


In [9]:
df[['AgeOfStore','SalesInThousands']].corr()

,AgeOfStore,SalesInThousands
AgeOfStore,1.000000,-0.028533
SalesInThousands,-0.028533,1.000000


### 2. ANOVA

In [11]:
#Separate sales data by promotion group
group1 = df[df['Promotion'] == 1]['SalesInThousands']
group2 = df[df['Promotion'] == 2]['SalesInThousands']
group3 = df[df['Promotion'] == 3]['SalesInThousands']


#Perform one-way ANOVA to test if there is a significant difference between the mean sales of the three promotion groups
f_stat, p_value = stats.f_oneway(group1, group2, group3)

print("F-statistic: ", f_stat)
print("p-value: ", p_value)

F-statistic:  21.953485793080677
p-value:  6.765849261408714e-10


- p-value < 0.05
- we reject the null hypothesis
- There is a statistically significant difference in mean sales across at least one promotion group

### 3. Tukey HSD (Post-hoc Test)

In [14]:
from statsmodels.stats.multicomp import pairwise_tukeyhsd

#Perform Tukey's Honest Significant Difference test
#This identifies which specific promotion pairs differ significantly

tukey = pairwise_tukeyhsd(
    endog=df['SalesInThousands'],
    groups=df['Promotion'],
    alpha=0.05
)

print(tukey)

 Multiple Comparison of Means - Tukey HSD, FWER=0.05 
group1 group2 meandiff p-adj   lower    upper  reject
-----------------------------------------------------
     1      2 -10.7696    0.0 -14.7738 -6.7654   True
     1      3  -2.7345 0.2444  -6.7388  1.2697  False
     2      3   8.0351    0.0   4.1208 11.9493   True
-----------------------------------------------------


***Promotion 1 vs 2***

- Promotion 1 generates $10.77k higher weekly sales

- Highly significant

***Promotion 2 vs 3***

- Promotion 3 generates $8.04k higher weekly sales

- Highly significant

***Promotion 1 vs 3***

- No statistically significant difference


Promotion 2 is clearly underperforming.


Promotion 1 and 3 perform similarily, but Promotion 1 has slightly hier mean sales.
- If cost is equal -> promotion 1
- If Promotion 3 cheaper -> promotion 3 is option

In [ ]:
#Effect Size (Eta Squared)

# Calculate overall mean
grand_mean = df['SalesInThousands'].mean()

# Calculate Sum of Squares Between (SSB)
ss_between = sum(
    len(df[df['Promotion'] == p]) *
    (df[df['Promotion'] == p]['SalesInThousands'].mean() - grand_mean) ** 2
    for p in df['Promotion'].unique()
)

# Calculate Total Sum of Squares (SST)
ss_total = sum((df['SalesInThousands'] - grand_mean) ** 2)

# Eta squared
eta_squared = ss_between / ss_total

print("Eta squared:", eta_squared)

Eta squared: 0.07455671898042318


We found:

Significant difference in weekly sales across promotions (ANOVA, p < 0.001)

Promotion 2 significantly underperforms compared to 1 and 3

Promotion 1 and 3 show no significant difference

Effect size indicates moderate practical impact (η² = 0.075)

#### Does the promotion effect remain significant after controlling for market characteristics?

In [16]:
import statsmodels.formula.api as smf

# Build regression model controlling for market size and store age
model = smf.ols(
    'SalesInThousands ~ C(Promotion) + C(MarketSize) + AgeOfStore',
    data=df
).fit()

print(model.summary())

                            OLS Regression Results                            
Dep. Variable:       SalesInThousands   R-squared:                       0.582
Model:                            OLS   Adj. R-squared:                  0.578
Method:                 Least Squares   F-statistic:                     150.9
Date:                Sun, 01 Mar 2026   Prob (F-statistic):          3.35e-100
Time:                        21:46:09   Log-Likelihood:                -2082.7
No. Observations:                 548   AIC:                             4177.
Df Residuals:                     542   BIC:                             4203.
Df Model:                           5                                         
Covariance Type:            nonrobust                                         
                              coef    std err          t      P>|t|      [0.025      0.975]
-------------------------------------------------------------------------------------------
Intercept                 

1. ANOVA confirms significant difference across campaigns (p < 0.001)

2. Tukey test shows Promotion 2 underperforms

3. Effect size is moderate (η² = 0.075)

4. Regression confirms Promotion 2 significantly lowers sales even after controlling for market size

5. Market size is the dominant driver of sales performance

## Conclusion

The evidence points to a pretty direct recommendation: Promotion 2 underperforms enough that it should not be scaled. Promotion 1 and Promotion 3 are close enough that the real decision between them would probably come down to execution cost or implementation complexity rather than raw sales lift.

What makes the analysis more useful than a simple significance test is the added context. Market size explains a meaningful share of the variation, so the conclusion is not just about which campaign won, but about how campaign results should be interpreted inside different store environments.